# CrisisQ — frozen-model external evaluation on new public data

**No retraining. No threshold tuning. No clinicians. No ZIP input.** Attach the **full output of your latest CrisisQ notebook** with Kaggle Add Input. This notebook recursively finds the expected frozen selection and models under `/kaggle/input`. Enable Internet; a GPU is unnecessary because the frozen INT8 and FP32 comparisons run on CPU as before.

**Scope:** new-source NLP evaluation using existing public binary text labels, a separate positive probe, a keyword-enriched labeled-negative set, and unlabeled emotional-support conversations. Sources are fixed public Git commits; sampling/exclusion rules are fixed before inference. The model-selection procedure and thresholds remain exactly those already frozen. New results are reporting only.

**Important independence limit with latest-output-only input:** the latest output may contain error texts, annotation samples, and mobile parity examples but not the complete original training/evaluation text archive. The notebook audits every available historical text and explicitly marks missing full splits. Passing this audit is **not proof of no historical contamination**. It is an external, no-retuning evaluation, with full independence still unverified if the historical archive is incomplete. The Twitter corpus is new to this workflow; unknown earlier author exposure and language-model pretraining exposure cannot be disproved by code.

**One Run All:** files, source revisions, sampling, model bytes, tokenizer files, and the evaluation plan are frozen before new-data inference. Re-running resumes the same frozen evaluation. Do not change models, thresholds, sampling budgets, source choice, exclusions or rules after looking at results. A new design after inspection requires a separately disclosed experiment and a new uninspected evaluation set.

Only final tables, probability caches, row identifiers/hashes, overlap reports and manifests are packaged. Public source text stays in `/kaggle/temp` and is not included in the saved results folder or download ZIP. Source labels are not clinical diagnoses; no medical readiness or safety guarantee is established.


In [ ]:
# Configuration — set before first execution; defaults match your verified frozen run.
from pathlib import Path
INPUT_BASE=Path('/kaggle/input')
RUN_ROOT_OVERRIDE=None  # optional path if duplicate attached outputs need disambiguation
EXPECTED_SELECTION_HASH='e976dbf922c37ce1406418ff9c5903a1ecc6a502aadb14d3d116ab18ef6ba626'
OUTPUT=Path('/kaggle/working/crisisq_external_public_v1')
PRIVATE=Path('/kaggle/temp/crisisq_external_public_v1')
PRIVATE.mkdir(parents=True,exist_ok=True)
ARMS=['ce','saqs']
INCLUDE_FIXED_R90=True  # evaluate all 10 seeds per arm at the previously saved R90 thresholds
# Primary inference uses every development-accepted seed, including comparison CE.
# The first accepted SAQS seed in the ORIGINAL seed order is the fixed primary configuration.
SAMPLE_SEED='2026-11-11-crisisq-external-v1'
TEST_PER_CLASS=500
PROBE_POSITIVES=300; PROBE_NEGATIVES=100
HARD_NEGATIVES=300
SUPPORT_CONVERSATIONS=300  # duplicate-linked conversation groups; actual turn count can exceed 600
NEAR_DUPLICATE_THRESHOLD=.90
N_BOOT=2000
# No 'fast smoke sample' switch: selecting a smaller set after viewing results is not permitted.
for d in ['manifests','tables','predictions','cache','figures']:
    (OUTPUT/d).mkdir(parents=True,exist_ok=True)


## 1. Locate the latest frozen run, then install matching inference libraries
Only data manifests are read at this stage. The input's saved source code is not executed. If the attached output is the reports package without models, execution stops with the missing-file explanation. No Hugging Face model is downloaded as a replacement.


In [ ]:
import json,hashlib,sys,subprocess,importlib.metadata,platform,os

def _digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
roots=[]
for selection in sorted(INPUT_BASE.rglob('selection_frozen.json')):
    root=selection.parent
    if RUN_ROOT_OVERRIDE and root.resolve()!=Path(RUN_ROOT_OVERRIDE).resolve(): continue
    p=root/'protocol_frozen.json'
    if not p.exists():continue
    try:
        s=json.loads(selection.read_text()); proto=json.loads(p.read_text())
        if _digest(s)==EXPECTED_SELECTION_HASH and s.get('protocol')==_digest(proto) and (root/'models').is_dir():
            roots.append((root,s,proto))
    except (ValueError,OSError):pass
if not roots:
    raise FileNotFoundError('Expected frozen run plus models/ not found under /kaggle/input. Attach the FULL latest notebook output, not the reports ZIP. No model will be retrained or substituted.')
RUN,SELECTION,PROTOCOL=roots[0]
assert not PROTOCOL.get('smoke',False), 'Smoke-test source models cannot produce publication results'
print('Frozen input:',RUN)
print('Original inference versions:',{k:PROTOCOL['versions'].get(k) for k in ['onnxruntime','transformers','onnx']})
# Install before importing ORT/Transformers. Preserve exact inference/tokenizer versions.
required={'numpy':None,'pandas':None,'scipy':None,'scikit-learn':None,
          'onnxruntime':PROTOCOL['versions']['onnxruntime'],
          'transformers':PROTOCOL['versions']['transformers'],'onnx':PROTOCOL['versions']['onnx']}
install=[]
for pkg,wanted in required.items():
    try:current=importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:current=None
    if current is None or (wanted is not None and current!=wanted):
        module={'scikit-learn':'sklearn','onnxruntime':'onnxruntime','transformers':'transformers','onnx':'onnx'}.get(pkg,pkg)
        if module in sys.modules:
            raise RuntimeError(f'{pkg} is already imported with a different version. Restart the kernel before Run All.')
        install.append(pkg+('=='+wanted if wanted else ''))
if install:
    print('Installing inference dependencies:',install)
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-cache-dir']+install)


In [ ]:
"""Audited inference-only utilities embedded in the notebook."""
from pathlib import Path
import json, hashlib, re, unicodedata, html, math, os, urllib.request, time, zipfile
import numpy as np
import pandas as pd
from scipy.stats import beta, t as student_t
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score


def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def freeze(path,obj):
    path=Path(path)
    if path.exists():
        if json.loads(path.read_text())!=json.loads(json.dumps(obj,default=str)):
            raise RuntimeError(f'{path.name} differs from the frozen run. Preserve the existing run; do not overwrite or retune after results.')
    else: atomic_json(path,obj)
    return digest(obj)

def normal_text(text):
    s=unicodedata.normalize('NFKC',html.unescape(str(text))).casefold()
    s=s.replace('\u2019',"'")
    return re.sub(r'\s+',' ',re.sub(r'[^\w\s]',' ',s)).strip()

def text_hash(text): return hashlib.sha256(normal_text(text).encode()).hexdigest()

def source_frame_digest(frame):
    # Reproduce the previous notebook's read_frame + frame_digest convention exactly.
    d=frame.copy()
    d['text']=d.text.astype(str); d['label']=pd.to_numeric(d.label,errors='raise').astype(int)
    oldhash=d.text.map(lambda s:hashlib.sha256(re.sub(r'\s+',' ',str(s)).strip().lower().encode()).hexdigest())
    if 'group_id' not in d: d['group_id']=d['group'].astype(str) if 'group' in d else oldhash
    for c,v in [('profile','real_world'),('source','unspecified')]:
        if c not in d: d[c]=v
    return digest(d[['text','label','group_id','profile','source']].to_dict('records'))

def discover_run(base,expected_hash):
    candidates=[]; seen=[]
    for selection in sorted(Path(base).rglob('selection_frozen.json')):
        root=selection.parent; protocol=root/'protocol_frozen.json'
        if not protocol.exists(): continue
        try:
            s=json.loads(selection.read_text()); p=json.loads(protocol.read_text())
            sh=digest(s); ph=digest(p)
            if s.get('protocol')!=ph or sh!=expected_hash: continue
            seen.append(str(root))
            if (root/'models').is_dir(): candidates.append((root,s,p,sh,ph))
        except (ValueError,OSError): continue
    if not candidates:
        raise FileNotFoundError('No matching frozen run with models/ found. Attach the FULL latest notebook output as Kaggle Input; the reports ZIP alone has no weights. Matching manifest roots seen: '+str(seen))
    # Duplicate mounts with identical frozen selection are safe; never select by evaluation performance.
    return candidates[0]


def build_gate(spec):
    # Rebuild from frozen data; never execute source code read from an attachment.
    required={'crisis','safe','negators','contrast','risk','v1_crisis','v1_safe'}
    if not required<=set(spec): raise ValueError('Frozen rule specification is incomplete')
    C=[(i,re.compile(p)) for i,c,p in spec['crisis']]
    S=[(i,re.compile(p)) for i,c,p in spec['safe']]
    negators=set(spec['negators']); contrast=re.compile(spec['contrast']); risk=re.compile(spec['risk'])
    def normalize(text):
        s=str(text).lower()
        for ch in ["'",'’','`','‘']: s=s.replace(ch,'')
        return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9\s]',' ',s)).strip()
    def route(text,version='v2',trace=False):
        s=normalize(text)
        if version=='v1':
            for p in spec['v1_safe']:
                if normalize(p) in s: return (0,'legacy_safe') if trace else 0
            for p in spec['v1_crisis']:
                if normalize(p) in s: return (1,'legacy_crisis') if trace else 1
            return (-1,'uncertain') if trace else -1
        aff=[]; neg=[]; safe=[]; spans=[]
        for rid,rx in C:
            for m in rx.finditer(s):
                (neg if any(w in negators for w in s[:m.start()].split()[-4:]) else aff).append(rid)
        for rid,rx in S:
            for m in rx.finditer(s): safe.append(rid); spans.append(m.span())
        if aff and safe: out,reason=-1,'conflict_to_stage2'
        elif aff: out,reason=1,'crisis'
        elif neg: out,reason=-1,'negated_crisis_to_stage2'
        elif safe:
            residual=s
            for a,b in sorted(spans,reverse=True): residual=residual[:a]+' '+residual[b:]
            if contrast.search(s) or risk.search(residual): out,reason=-1,'safe_with_residual_risk_to_stage2'
            else: out,reason=0,'safe_context'
        else: out,reason=-1,'uncertain'
        return (out,reason) if trace else out
    return route


def download_pinned(repo,commit,name,directory):
    from urllib.parse import quote
    dest=Path(directory)/(repo.replace('/','__')+'__'+Path(name).name)
    dest.parent.mkdir(parents=True,exist_ok=True)
    url=f'https://raw.githubusercontent.com/{repo}/{commit}/'+quote(name)
    if not dest.exists():
        tmp=dest.with_suffix(dest.suffix+'.download')
        for attempt in range(3):
            try:
                req=urllib.request.Request(url,headers={'User-Agent':'CrisisQ-external-evaluation/1.0'})
                with urllib.request.urlopen(req,timeout=90) as r, tmp.open('wb') as f:
                    while True:
                        b=r.read(1024*1024)
                        if not b: break
                        f.write(b)
                tmp.replace(dest); break
            except Exception:
                if attempt==2: raise
                time.sleep(2)
    return dest,dict(repo=repo,commit=commit,path=name,url=url,sha256=file_sha(dest),bytes=dest.stat().st_size)


def parse_twitter(path):
    d=pd.read_csv(path,keep_default_na=False)
    if not {'tweet','intention'}<=set(d): raise ValueError('Public Twitter schema changed: expected tweet,intention')
    labels=pd.to_numeric(d.intention,errors='raise')
    if not labels.isin([0,1]).all(): raise ValueError('Unexpected Twitter labels; do not guess a mapping')
    return pd.DataFrame(dict(text=d.tweet.astype(str),label=labels.astype(int),source='twitter_intention',
        source_id=['twitter:'+str(i) for i in range(len(d))],group_id=['twitter:'+str(i) for i in range(len(d))],role='author'))


def parse_esconv(path):
    rows=[]; data=json.loads(Path(path).read_text())
    if not isinstance(data,list): raise ValueError('ESConv must be a list')
    for i,item in enumerate(data):
        dialog=item.get('dialog')
        if not isinstance(dialog,list): raise ValueError('ESConv dialog schema mismatch')
        for role in ['seeker','supporter']:
            # One first substantive utterance per role per conversation; not model-based selection.
            chosen=None
            for j,turn in enumerate(dialog):
                if turn.get('speaker')!=role: continue
                text=turn.get('content')
                if not isinstance(text,str): raise ValueError('Expected ESConv content text')
                if len(text.strip())>=30 and len(text.split())>=5:
                    chosen=(j,text); break
            if chosen:
                j,text=chosen
                rows.append(dict(text=text,label=-1,source='esconv',source_id=f'esconv:{i}:{j}',group_id=f'esconv:{i}',role=role))
    if not rows: raise ValueError('No ESConv turns parsed')
    return pd.DataFrame(rows)


def collect_old_texts(root,protocol):
    root=Path(root); frames=[]; inventory=[]; verified={}
    for p in sorted(root.rglob('*.csv')):
        if p.stat().st_size>150*1024**2: continue
        try:
            d=pd.read_csv(p,keep_default_na=False)
            col=next((c for c in ['text','tweet','content'] if c in d.columns),None)
            if col:
                vals=d[col].astype(str); vals=vals[vals.str.len()>=10]
                frames.append(pd.DataFrame(dict(text=vals,origin=str(p.relative_to(root)))))
                inventory.append(dict(file=str(p.relative_to(root)),rows=len(vals)))
            if col=='text' and 'label' in d:
                h=source_frame_digest(d)
                for split,wanted in protocol.get('splits',{}).items():
                    if h==wanted: verified[split]=str(p.relative_to(root))
        except (ValueError,TypeError,KeyError,pd.errors.ParserError): continue
    old=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['text','origin'])
    if len(old): old=old.drop_duplicates('text').reset_index(drop=True)
    expected=set(protocol.get('splits',{})); missing=sorted(expected-set(verified))
    return old,dict(verified_complete_splits=verified,missing_complete_splits=missing,
        complete_historical_archive=bool(expected) and not missing,known_unique_texts=len(old),files=inventory,
        conclusion='complete exact-text reference coverage; cross-post/author and pretraining overlap still not proven' if not missing else 'partial historical overlap audit: absence of overlap does not prove no training/evaluation contamination')


def deduplicate_and_audit(candidates,old,near_threshold=.90):
    d=candidates.copy(); d['text']=d.text.astype(str)
    d=d[d.text.str.strip().str.len().ge(20)&d.text.str.split().str.len().ge(3)].copy()
    d['text_hash']=d.text.map(text_hash)
    conflicts=d[d.label>=0].groupby('text_hash').label.nunique()
    conflicting=set(conflicts[conflicts>1].index)
    d=d[~d.text_hash.isin(conflicting)].copy()
    # Keep one deterministic representative per text; preserve all its original conversation links.
    d=d.sort_values('source_id').reset_index(drop=True)
    all_group_links=d.groupby('text_hash').group_id.apply(list).to_dict()
    d=d.drop_duplicates('text_hash').reset_index(drop=True)
    n=len(d); parent=list(range(n))
    def find(a):
        while parent[a]!=a:
            parent[a]=parent[parent[a]];a=parent[a]
        return a
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[max(a,b)]=min(a,b)
    first={}
    for i,row in d.iterrows():
        for g in all_group_links[row.text_hash]:
            if g in first: union(i,first[g])
            else: first[g]=i
    oldtexts=old.text.astype(str).tolist() if len(old) else []
    oldhash=set(map(text_hash,oldtexts))
    exact=d.text_hash.isin(oldhash).to_numpy(); old_match={}; near_edges=0
    texts=oldtexts+d.text.tolist()
    if not texts: raise ValueError('No public evaluation texts')
    vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=100000,dtype=np.float32)
    z=vec.fit_transform(texts); oldz=z[:len(oldtexts)]; newz=z[len(oldtexts):]
    for start in range(0,n,96):
        block=newz[start:start+96]
        if len(oldtexts):
            sim=(block@oldz.T).tocsr()
            for local in range(block.shape[0]):
                row=sim.getrow(local)
                if row.nnz and row.data.max()>=near_threshold:
                    j=int(row.indices[row.data.argmax()]);old_match[start+local]=(j,float(row.data.max()))
        sim=(block@newz.T).tocoo()
        for a,b,v in zip(sim.row,sim.col,sim.data):
            i=start+int(a); j=int(b)
            if j>i and v>=near_threshold: union(i,j);near_edges+=1
    roots=[find(i) for i in range(n)]
    members={}
    for i,r in enumerate(roots): members.setdefault(r,[]).append(i)
    groupnames={r:'cluster:'+min(d.text_hash.iloc[i] for i in ii) for r,ii in members.items()}
    d['group_id']=[groupnames[r] for r in roots]
    contaminated={roots[i] for i in range(n) if exact[i] or i in old_match}
    flags=[]
    for i in range(n):
        if exact[i] or i in old_match:
            flags.append(dict(source_id=d.source_id.iloc[i],text_hash=d.text_hash.iloc[i],exact_known_overlap=bool(exact[i]),
                lexical_similarity=old_match.get(i,(None,None))[1],cluster=d.group_id.iloc[i]))
    removed=d[d.group_id.isin({groupnames[r] for r in contaminated})]
    clean=d.drop(removed.index).reset_index(drop=True)
    return clean,pd.DataFrame(flags,columns=['source_id','text_hash','exact_known_overlap','lexical_similarity','cluster']),dict(
        input_rows=len(candidates),label_conflict_hashes=len(conflicting),unique_eligible_before_overlap=n,
        exact_known_overlap_rows=int(exact.sum()),lexical_known_overlap_rows=len(old_match),
        candidate_near_duplicate_edges=near_edges,removed_rows_in_overlapping_groups=len(removed),remaining_rows=len(clean))


def draw_groups(pool,budgets,salt):
    # Whole duplicate/conversation groups go to one evaluation partition.
    rank=lambda g:hashlib.sha256((salt+'|'+str(g)).encode()).hexdigest()
    target={int(k):int(v) for k,v in budgets.items()}; counts={k:0 for k in target}; picked=[]
    for group in sorted(pool.group_id.unique(),key=rank):
        g=pool[pool.group_id==group]
        if not any(counts.get(int(y),0)<target.get(int(y),0) for y in g.label.unique()): continue
        picked.append(g)
        for y,c in g.label.value_counts().items(): counts[int(y)]=counts.get(int(y),0)+int(c)
        if all(counts.get(y,0)>=n for y,n in target.items()): break
    return (pd.concat(picked,ignore_index=True) if picked else pool.iloc[:0].copy()),counts


def metric(y,pred):
    y=np.asarray(y); pred=np.asarray(pred); valid=np.isin(y,[0,1]); pos=y==1; neg=y==0
    npos=int(pos.sum()); nneg=int(neg.sum()); tp=int(((pred==1)&pos).sum()); fp=int(((pred==1)&neg).sum())
    return dict(n=len(y),n_labeled=int(valid.sum()),n_positive=npos,n_negative=nneg,tp=tp,fn=npos-tp,fp=fp,tn=nneg-fp,
        recall=tp/npos if npos else None,fpr=fp/nneg if nneg else None,
        alert_rate=float(np.mean(pred==1)) if len(pred) else None,
        precision=tp/(tp+fp) if valid.any() and tp+fp else None)

def decide(p,tau,gate=None):
    out=(np.asarray(p)>=float(tau)).astype(int)
    return out if gate is None else np.where(np.asarray(gate)<0,out,gate).astype(int)

def exact_bounds(k,n):
    if not n: return None,None,None
    lo=0. if not k else float(beta.ppf(.025,k,n-k+1))
    hi=1. if k==n else float(beta.ppf(.975,k+1,n-k))
    upper=1. if k==n else float(beta.ppf(.95,k+1,n-k))
    return lo,hi,upper

def group_interval(values,groups,B=2000,seed=20261111):
    values=np.asarray(values,float); groups=np.asarray(groups)
    if not len(values): return None,None
    names=np.unique(groups)
    if len(names)<2: return None,None
    counts=np.array([(groups==g).sum() for g in names],float)
    sums=np.array([values[groups==g].sum() for g in names],float)
    rng=np.random.default_rng(seed); stats=[]
    for start in range(0,B,128):
        idx=rng.integers(0,len(names),size=(min(128,B-start),len(names)))
        stats.extend((sums[idx].sum(axis=1)/counts[idx].sum(axis=1)).tolist())
    return tuple(map(float,np.quantile(stats,[.025,.975])))

def paired_seed_interval(values):
    v=np.asarray(values,float);v=v[np.isfinite(v)]
    if not len(v): return None,None,None
    mean=float(v.mean())
    if len(v)<2: return mean,None,None
    half=float(student_t.ppf(.975,len(v)-1)*v.std(ddof=1)/np.sqrt(len(v)))
    return mean,mean-half,mean+half


def model_files(root,arm,seed,variant):
    directory=Path(root)/'models'/f'{arm}__{seed}'; model=directory/f'{variant}.onnx'
    if not model.exists(): raise FileNotFoundError(f'Missing frozen weight file {model}; no retraining or model substitution is allowed.')
    if not (directory/'tokenizer_config.json').exists() or not ((directory/'tokenizer.json').exists() or (directory/'vocab.txt').exists()):
        raise FileNotFoundError(f'Local tokenizer is missing in {directory}')
    import onnx
    graph=onnx.load(str(model),load_external_data=False)
    paths=[model]
    for tensor in graph.graph.initializer:
        if tensor.data_location==onnx.TensorProto.EXTERNAL:
            loc=next((e.value for e in tensor.external_data if e.key=='location'),None)
            if not loc: raise ValueError('External ONNX tensor has no location')
            p=(directory/loc).resolve()
            if directory.resolve() not in p.parents: raise ValueError('Unsafe external weight path')
            if not p.exists(): raise FileNotFoundError(f'Missing external weights: {p}')
            paths.append(p)
    paths.extend(p for p in directory.iterdir() if p.name in ['tokenizer_config.json','tokenizer.json','vocab.txt','special_tokens_map.json','config.json','added_tokens.json'])
    return directory,sorted(set(paths),key=str)


def infer_checkpointed(model_path,tokenizer_dir,encoded,path,signature,max_length,threads=4):
    import onnxruntime as ort
    path=Path(path); meta=path.with_suffix('.json'); shards=path.parent/(path.stem+'_shards')
    cache_meta=dict(signature=signature,n=len(encoded['input_ids']),batch=1,max_length=max_length)
    if path.exists():
        if not meta.exists() or json.loads(meta.read_text())!=cache_meta: raise RuntimeError('Prediction cache identity mismatch')
        p=np.load(path)
        if p.shape!=(cache_meta['n'],) or not np.isfinite(p).all(): raise RuntimeError('Corrupt probability cache')
        return p
    shards.mkdir(parents=True,exist_ok=True); freeze(shards/'identity.json',cache_meta)
    so=ort.SessionOptions(); so.intra_op_num_threads=threads
    session=ort.InferenceSession(str(model_path),sess_options=so,providers=['CPUExecutionProvider'])
    ins=session.get_inputs();names={i.name for i in ins}
    if not {'input_ids','attention_mask'}<=names or names-{'input_ids','attention_mask','token_type_ids'}:
        raise ValueError(f'Unexpected ONNX inputs: {names}')
    for i in ins:
        if i.type!='tensor(int64)': raise ValueError(f'Expected int64 input, got {i.type}')
        if isinstance(i.shape[1],int) and i.shape[1]!=max_length: raise ValueError('Frozen ONNX sequence length differs from protocol')
    outputs=session.get_outputs()
    if 'logits' not in {o.name for o in outputs}: raise ValueError('Expected named logits output')
    allp=[]; n=cache_meta['n']
    for start in range(0,n,128):
        end=min(start+128,n); chunk=shards/f'{start:08d}.npy'
        if chunk.exists():
            p=np.load(chunk)
            if p.shape!=(end-start,) or not np.isfinite(p).all(): raise RuntimeError(f'Corrupt shard {chunk}')
        else:
            vals=[]
            for i in range(start,end):
                feed={k:encoded[k][i:i+1].astype(np.int64) for k in names if k in encoded}
                if 'token_type_ids' in names and 'token_type_ids' not in feed: feed['token_type_ids']=np.zeros_like(feed['input_ids'])
                logits=np.asarray(session.run(['logits'],feed)[0],dtype=np.float64)
                if logits.shape!=(1,2): raise ValueError(f'Expected two logits with positive class index 1; got {logits.shape}')
                logits-=logits.max(axis=1,keepdims=True); e=np.exp(logits); vals.append(float((e/e.sum(axis=1,keepdims=True))[0,1]))
            p=np.asarray(vals); tmp=chunk.with_suffix('.tmp.npy');np.save(tmp,p);tmp.replace(chunk)
        allp.extend(p.tolist())
        if start%512==0: print(f'  inference {end}/{n}',flush=True)
    p=np.asarray(allp)
    if not np.isfinite(p).all() or ((p<0)|(p>1)).any(): raise RuntimeError('Invalid probabilities')
    tmp=path.with_suffix('.tmp.npy');np.save(tmp,p);tmp.replace(path);atomic_json(meta,cache_meta)
    del session
    return p


In [ ]:
import onnxruntime as ort,onnx,gc
from transformers import AutoTokenizer
for pkg in ['onnxruntime','transformers','onnx']:
    if importlib.metadata.version(pkg)!=PROTOCOL['versions'][pkg]:
        raise RuntimeError(f'{pkg} mismatch. Restart kernel and rerun dependency cell.')
MAX_LENGTH=int(PROTOCOL['max_length'])
if int(PROTOCOL['inference_batch'])!=1:
    raise RuntimeError('This notebook expects your frozen single-message inference protocol.')
THREADS=min(4,os.cpu_count() or 1)
RUNTIME={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','onnx','onnxruntime','transformers']}
RUNTIME.update(dict(python=sys.version,platform=platform.platform(),processor=platform.processor(),
                    cpu_count=os.cpu_count(),threads=THREADS,provider='CPUExecutionProvider',batch=1,
                    ort_build=ort.get_build_info(),historical_cpu=PROTOCOL.get('environment')))
GATE=build_gate(PROTOCOL['rules'])
# Guard against accidental changes in the reconstruction of the frozen routing policy.
for text,expected in [('I am not suicidal.',0),('I want to end my life.',1),
    ('I was not suicidal yesterday, but now I want to die.',-1)]:
    if GATE(text)!=expected:raise RuntimeError('Frozen gate reconstruction regression failed')
print('Frozen selection verified. CPU provider:',ort.get_available_providers())


## 2. Find available historical text and inventory the actual model files
The latest output only is used. Every requested weight file and tokenizer must exist. Historical coverage is tested against the old full-frame digests; an error sample is never treated as the full training archive. Missing full-text splits are recorded and do not silently become a claim of independent evaluation.


In [ ]:
OLD,HISTORY=collect_old_texts(RUN,PROTOCOL)
atomic_json(OUTPUT/'manifests/historical_coverage.json',HISTORY)
print('Historical unique texts available:',HISTORY['known_unique_texts'])
print('Missing complete historical splits:',HISTORY['missing_complete_splits'])
print('Independence qualification:',HISTORY['conclusion'])
ITEMS=[item for item in SELECTION['models'].values() if item['arm'] in ARMS]
if not ITEMS:raise ValueError('No requested arms in the frozen selection')
for arm in ARMS:
    chosen=[i for i in ITEMS if i['arm']==arm]
    if not chosen:raise ValueError(f'Frozen selection has no arm {arm}')
    expected_seeds=PROTOCOL['seeds'] if arm in ['ce','saqs'] else PROTOCOL['factorial_seeds']
    if {i['seed'] for i in chosen}!=set(expected_seeds):raise ValueError('Incomplete frozen seed manifest')
JOBS={};ANALYSES=[];STATUS=[]
def add_job(arm,seed,variant):
    key=f'{arm}__{seed}__{variant}'
    if key not in JOBS:
        directory,paths=model_files(RUN,arm,seed,variant)
        JOBS[key]=dict(arm=arm,seed=seed,variant=variant,directory=str(directory),model_path=str(directory/f'{variant}.onnx'),
            files={str(p.relative_to(RUN)):file_sha(p) for p in paths})
    return key
for item in ITEMS:
    arm,seed=item['arm'],item['seed'];sel=item['selected']
    STATUS.append(dict(arm=arm,seed=seed,development_accepted=sel is not None,
        variant=None if sel is None else sel['variant'],tau=None if sel is None else sel['tau']))
    if sel:
        key=add_job(arm,seed,sel['variant']);add_job(arm,seed,'fp32')
        for system in ['neural','hybrid_v1','hybrid_v2']:
            ANALYSES.append(dict(job=key,arm=arm,seed=seed,variant=sel['variant'],analysis='source_selected',system=system,tau=sel['tau']))
    if INCLUDE_FIXED_R90:
        for variant in ['fp32','qint8']:
            key=add_job(arm,seed,variant)
            for system in ['neural','hybrid']:
                tau=item.get('matched_R90',{}).get(f'{variant}|{system}')
                if tau is None:raise ValueError(f'Missing frozen R90 threshold: {arm}/{seed}/{variant}/{system}')
                ANALYSES.append(dict(job=key,arm=arm,seed=seed,variant=variant,analysis='source_R90',system=system,tau=tau))
pd.DataFrame(STATUS).to_csv(OUTPUT/'tables/source_development_status.csv',index=False)
original_order=PROTOCOL['seeds']
primary=next((i for seed in original_order for i in ITEMS if i['arm']=='saqs' and i['seed']==seed and i['selected']),None)
PRIMARY=None if primary is None else dict(arm='saqs',seed=primary['seed'],**primary['selected'])
print('Fixed primary configuration:',PRIMARY)
print('Unique frozen ONNX graphs to evaluate:',len(JOBS))
print('Models are read directly from Kaggle Input; they are never copied, trained, exported or quantized.')


## 3. Download two fixed public sources and construct disjoint evaluation partitions

* **Twitter suicidal-intention dataset:** 9,119 published tweet rows, with `tweet` and `intention` columns. Use the existing mapping 1=suicidal, 0=non-suicidal. The repository does not establish clinician validation; these are source task labels. [Original repository](https://github.com/laxmimerit/twitter-suicidal-intention-dataset).
* **ESConv:** use the original repository's support conversations, one first substantive seeker and supporter turn per conversation. It has no crisis labels, so output **alert rates only**, never FPR/recall. [Original repository and citation](https://github.com/thu-coai/Emotional-Support-Conversation), [dataset card](https://huggingface.co/datasets/thu-coai/esconv). Academic research use; preserve source terms and attribution.

The notebook does not treat a counselor answer or absence of a keyword as proof of a negative label. The labeled hard-negative subset is formed only from Twitter rows already labeled 0, enriched with a fixed language pattern independently of model scores. The probe uses different positive/negative groups from the test and hard-negative subsets. All three labeled partitions come from the same Twitter corpus and are not three independent source corpora.

Rows with conflicting duplicate labels are excluded before predictions. Known historical exact/lexical overlaps remove the entire corresponding new duplicate/conversation group. Texts are not rewritten for inference; normalization is for overlap checks only. Sampling is deterministic, group-disjoint, model-score-independent, and frozen before inference. If a target cannot be met, actual counts and the shortfall are reported without synthetic filling or retuning.


In [ ]:
SOURCES={
 'twitter':dict(repo='laxmimerit/twitter-suicidal-intention-dataset',commit='d800024117181614aa908c821400cede24e07355',path='twitter-suicidal_data.csv',expected_sha256='e2abd725b4f1c3d758e4df0c9f04416954ac96d9be61dc25a6c73cdaeecc470e'),
 'esconv':dict(repo='thu-coai/Emotional-Support-Conversation',commit='f262d062ad74cb39b17ea476facc81568ddcba24',path='ESConv.json',expected_sha256='aa0556c5b330562ba009c1cd5137486bfa2a7255f33225a6524cd58f7efdd9af')}
SOURCE_RECORDS={};source_paths={}
for name,spec in SOURCES.items():
    p,meta=download_pinned(spec['repo'],spec['commit'],spec['path'],PRIVATE)
    if meta['sha256']!=spec['expected_sha256']:
        raise RuntimeError(f'Pinned public source byte hash mismatch: {name}. Do not replace it silently.')
    SOURCE_RECORDS[name]=meta;source_paths[name]=p
TWITTER=parse_twitter(source_paths['twitter']);SUPPORT=parse_esconv(source_paths['esconv'])
RAW=pd.concat([TWITTER,SUPPORT],ignore_index=True)
CLEAN,OVERLAP_FLAGS,AUDIT=deduplicate_and_audit(RAW,OLD,NEAR_DUPLICATE_THRESHOLD)
OVERLAP_FLAGS.to_csv(OUTPUT/'tables/overlap_exclusions.csv',index=False)
atomic_json(OUTPUT/'manifests/overlap_audit.json',AUDIT)
print('Source binary label counts:',TWITTER.label.value_counts().to_dict())
print('Overlap/duplicate audit:',AUDIT)

STRESS_PATTERN=r'\b(?:suicid\w*|self[ -]?harm\w*|kill|killing|die|dying|dead|death|overdos\w*|hopeless|worthless|burden|counsel\w*|therap\w*|depress\w*)\b'
EXPLICIT_PATTERN=r'\b(?:suicid\w*|kill(?:ing)? myself|end(?:ing)? my (?:own )?life|take my (?:own )?life|want to die|self[ -]?harm\w*|overdos\w*)\b'
stress=re.compile(STRESS_PATTERN,re.I);explicit=re.compile(EXPLICIT_PATTERN,re.I)
sets={};used=set();shortfalls=[]
def take(name,pool,budgets):
    pool=pool[~pool.group_id.isin(used)]
    d,actual=draw_groups(pool,budgets,SAMPLE_SEED+'|'+name)
    if not len(d):raise RuntimeError(f'No eligible groups for {name}; report unavailable, do not substitute another source silently.')
    used.update(d.group_id);d=d.copy();d['partition']=name;sets[name]=d
    shortfalls.append(dict(partition=name,target=budgets,actual=actual,unmet={y:n-actual.get(y,0) for y,n in budgets.items() if actual.get(y,0)<n}))
tw=CLEAN[CLEAN.source=='twitter_intention']
take('labeled_hard_negative',tw[(tw.label==0)&tw.text.map(lambda s:bool(stress.search(s)))],{0:HARD_NEGATIVES})
take('positive_probe',tw,{1:PROBE_POSITIVES,0:PROBE_NEGATIVES})
take('binary_test',tw,{1:TEST_PER_CLASS,0:TEST_PER_CLASS})
es=CLEAN[(CLEAN.source=='esconv')&~CLEAN.group_id.isin(used)]
# Keep paired seeker/supporter coverage; linked duplicate groups may contain >1 conversation.
paired=es.groupby('group_id').role.nunique();es=es[es.group_id.isin(paired[paired==2].index)]
rank=lambda g:hashlib.sha256((SAMPLE_SEED+'|support|'+str(g)).encode()).hexdigest()
selected_groups=sorted(es.group_id.unique(),key=rank)[:SUPPORT_CONVERSATIONS]
if not selected_groups:raise RuntimeError('No eligible paired ESConv groups remain')
sets['support_unlabeled']=es[es.group_id.isin(selected_groups)].assign(partition='support_unlabeled').copy()
E=pd.concat(sets.values(),ignore_index=True)
# Each duplicate/conversation group must belong to exactly one partition.
assert E.groupby('group_id').partition.nunique().max()==1
assert not E.text_hash.duplicated().any()
E['explicit_keyword_present']=E.text.map(lambda s:bool(explicit.search(s)))
E['gate_v1']=[GATE(t,'v1') for t in E.text]
E['gate_v2']=[GATE(t,'v2') for t in E.text]
E['route_reason']=[GATE(t,'v2',trace=True)[1] for t in E.text]
# Stable order is part of every prediction cache key and the frozen evaluation manifest.
E=E.sort_values(['partition','source_id']).reset_index(drop=True)
public_index=E.drop(columns=['text'])
public_index.to_csv(OUTPUT/'tables/evaluation_row_index.csv',index=False)
E.to_csv(PRIVATE/'frozen_evaluation_texts.csv',index=False) # LOCAL ONLY, excluded from ZIP
atomic_json(OUTPUT/'manifests/sample_shortfalls.json',shortfalls)
print(E.groupby(['partition','source','role','label']).size().to_string())
print('Target shortfalls:',[x for x in shortfalls if x['unmet']])
print('Labels 0/1 come from the Twitter source; label -1 means unknown, not negative.')


## 4. Freeze the complete new evaluation before opening predictions
The manifests bind original protocol/selection hashes, all weight and tokenizer hashes, actual selected rows/labels/groups, source bytes, the inference environment and this implementation. All model thresholds are copied from the original selection. No threshold is learned on the new data. The same recorded models remain selected even if their new-data performance is poor.


In [ ]:
EVALUATION_HASH=digest(E[['source_id','text_hash','label','group_id','partition','role']].to_dict('records'))
EVAL_PLAN=dict(version='external_public_v1',implementation_hash='8e6fbbc59fa68c5003b8d9ecee36e4b5ef6c40c8ad41500af9fe8cda8557cd3c',
 source_protocol_hash=digest(PROTOCOL),source_selection_hash=digest(SELECTION),primary=PRIMARY,
 source_records=SOURCE_RECORDS,historical_coverage=HISTORY,overlap_audit=AUDIT,
 configuration=dict(arms=ARMS,include_fixed_r90=INCLUDE_FIXED_R90,sampling_seed=SAMPLE_SEED,
   test_per_class=TEST_PER_CLASS,probe_positives=PROBE_POSITIVES,probe_negatives=PROBE_NEGATIVES,
   hard_negatives=HARD_NEGATIVES,support_conversations=SUPPORT_CONVERSATIONS,
   near_duplicate_threshold=NEAR_DUPLICATE_THRESHOLD,stress_pattern=STRESS_PATTERN,
   explicit_pattern=EXPLICIT_PATTERN,N_BOOT=N_BOOT),
 evaluation_rows=EVALUATION_HASH,evaluation_counts=E.groupby('partition').size().to_dict(),
 jobs={k:{c:v for c,v in job.items() if c not in ['directory','model_path']} for k,job in JOBS.items()},
 analyses=ANALYSES,runtime=RUNTIME,max_length=MAX_LENGTH,padding='max_length',batch=1,
 independence_status='new-source external no-retuning evaluation; historical contamination remains incompletely auditable' if not HISTORY['complete_historical_archive'] else 'new-source external no-retuning evaluation with complete available-text audit; no claim about pretraining or hidden author identity')
PLAN_HASH=freeze(OUTPUT/'manifests/evaluation_frozen.json',EVAL_PLAN)
print('NEW EVALUATION PLAN FROZEN:',PLAN_HASH)
print(EVAL_PLAN['independence_status'])


## 5. Run the frozen models, with resumable probability caches
All files are read directly from `/kaggle/input`. Each ONNX graph is evaluated only once on the fixed new rows; multiple systems/threshold comparisons then use that cache. Batch=1, fixed padding, frozen sequence length, original ORT/Transformers versions and CPU inference are preserved. CPU hardware can still differ; the environment is recorded. Expect hours of CPU inference when all seed-paired R90 controls are enabled. No GPU training is performed.

When the source output has a compatible `mobile/*_parity.csv`, a small set of its previously reported probabilities is checked first for that exact model. A mismatch stops the run before its new-data inference. If no compatible reference exists for a model, numerical runtime parity is explicitly **unverified** rather than fabricated.


In [ ]:
PROBS={};ENCODED={};PARITY=[];TOKEN_LENGTHS={}
for number,(key,job) in enumerate(sorted(JOBS.items()),1):
    print(f'Model {number}/{len(JOBS)}: {key}',flush=True)
    directory=Path(job['directory']);model_path=Path(job['model_path'])
    tokfiles={k:v for k,v in job['files'].items() if Path(k).name in ['tokenizer.json','tokenizer_config.json','vocab.txt','special_tokens_map.json','added_tokens.json']}
    token_key=digest({Path(k).name:v for k,v in tokfiles.items()})
    tok=AutoTokenizer.from_pretrained(str(directory),local_files_only=True,trust_remote_code=False)
    if token_key not in ENCODED:
        full=tok(E.text.tolist(),padding=False,truncation=False,return_length=True)
        TOKEN_LENGTHS[token_key]=np.asarray(full['length'])
        enc=tok(E.text.tolist(),padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='np')
        ENCODED[token_key]={k:np.asarray(v,dtype=np.int64) for k,v in enc.items() if k in ['input_ids','attention_mask','token_type_ids']}
    parity_status='not_available'
    mobile_manifest=RUN/'mobile/protocol.json'
    if mobile_manifest.exists():
        mm=json.loads(mobile_manifest.read_text())
        if (mm.get('arm'),mm.get('seed'),mm.get('variant'))==(job['arm'],job['seed'],job['variant']):
            for parity_file in sorted((RUN/'mobile').glob('*_parity.csv')):
                reference=pd.read_csv(parity_file,keep_default_na=False)
                if not {'text','cpu_probability'}<=set(reference):continue
                reference=reference.head(16)
                pe=tok(reference.text.tolist(),padding='max_length',truncation=True,max_length=MAX_LENGTH,return_tensors='np')
                signature=digest(dict(plan=PLAN_HASH,job=key,parity_file=file_sha(parity_file)))
                measured=infer_checkpointed(model_path,directory,pe,OUTPUT/'cache'/f'parity__{key}__{parity_file.stem}.npy',signature,MAX_LENGTH,THREADS)
                expected=pd.to_numeric(reference.cpu_probability).to_numpy()
                err=float(np.max(np.abs(measured-expected)))
                if not np.allclose(measured,expected,atol=1e-5,rtol=1e-5):
                    raise RuntimeError(f'CPU reference parity failed for {key}, max probability difference={err}. Preserve the run and investigate runtime differences; do not retune.')
                parity_status='verified_against_prior_CPU_reference'
                PARITY.append(dict(job=key,reference=parity_file.name,n=len(reference),max_abs_error=err,status=parity_status))
    if parity_status=='not_available':PARITY.append(dict(job=key,reference=None,n=0,max_abs_error=None,status=parity_status))
    signature=digest(dict(plan=PLAN_HASH,job=key,model_files=job['files'],tokenizer=token_key))
    PROBS[key]=infer_checkpointed(model_path,directory,ENCODED[token_key],OUTPUT/'predictions'/f'{key}.npy',signature,MAX_LENGTH,THREADS)
    del tok;gc.collect()
pd.DataFrame(PARITY).to_csv(OUTPUT/'tables/runtime_parity.csv',index=False)
truncation=[]
for tokenizer_key,lengths in TOKEN_LENGTHS.items():
    for part in E.partition.unique():
        mask=E.partition.eq(part).to_numpy()
        truncation.append(dict(tokenizer_key=tokenizer_key,partition=part,n=int(mask.sum()),
            token_truncation_rate=float((lengths[mask]>MAX_LENGTH).mean()),max_original_tokens=int(lengths[mask].max())))
pd.DataFrame(truncation).to_csv(OUTPUT/'tables/truncation_audit.csv',index=False)
print('Frozen-model inference complete. No models or thresholds were changed.')


## 6. Report absolute performance, routing effects, and compression transitions
Dataset-label FPR is reported only on source-labeled negatives. ESConv uses alert rate exclusively. Precision on the stratified Twitter test/probe reflects the constructed sample prevalence and is not an estimate of deployment precision. The keyword-enriched negative subset is a targeted stress test, not population FPR.

Confidence intervals resample duplicate/conversation groups. Missing Twitter author IDs prevent checking repeat authors; duplicate groups are an imperfect independence proxy. Exact binomial bounds in the flip table are descriptive per-seed bounds and should not replace group-aware interpretation. Seeds are never multiplied by examples to manufacture independent safety trials.


In [ ]:
RESULTS=[];PREDICTIONS={};ROUTING=[];FLIPS=[];SUBGROUPS=[]
y=E.label.to_numpy();groups=E.group_id.to_numpy()
partition_indices={name:np.flatnonzero(E.partition.to_numpy()==name) for name in sets}
def get_gate(system):
    if system=='neural':return None
    return E.gate_v1.to_numpy() if system=='hybrid_v1' else E.gate_v2.to_numpy()
for a in ANALYSES:
    gate=get_gate(a['system']);pr=decide(PROBS[a['job']],a['tau'],gate)
    ident=(a['arm'],a['seed'],a['variant'],a['analysis'],a['system']);PREDICTIONS[ident]=pr
    for name,idx in partition_indices.items():
        yy=y[idx];pp=pr[idx];m=metric(yy,pp)
        if (yy<0).all():
            alert_lo,alert_hi=group_interval(pp,groups[idx],N_BOOT)
            rec_lo=rec_hi=fpr_lo=fpr_hi=None
        else:
            pos=idx[y[idx]==1];neg=idx[y[idx]==0]
            rec_lo,rec_hi=group_interval(pr[pos],groups[pos],N_BOOT) if len(pos) else (None,None)
            fpr_lo,fpr_hi=group_interval(pr[neg],groups[neg],N_BOOT) if len(neg) else (None,None)
            alert_lo,alert_hi=group_interval(pp,groups[idx],N_BOOT)
        RESULTS.append(dict(**a,partition=name,primary=(PRIMARY is not None and a['analysis']=='source_selected' and a['arm']==PRIMARY['arm'] and a['seed']==PRIMARY['seed'] and a['variant']==PRIMARY['variant'] and a['system']=='hybrid_v2'),
            stage2_rate=1. if gate is None else float((gate[idx]<0).mean()),
            recall_group_lo=rec_lo,recall_group_hi=rec_hi,fpr_group_lo=fpr_lo,fpr_group_hi=fpr_hi,
            alert_group_lo=alert_lo,alert_group_hi=alert_hi,**m))
    # Descriptive slices are predefined labels on naturally occurring public text, not new annotations.
    slices={
      'labeled_positive_with_explicit_keywords':(y==1)&E.explicit_keyword_present.to_numpy(),
      'labeled_positive_without_prespecified_explicit_keywords':(y==1)&~E.explicit_keyword_present.to_numpy(),
      'public_rule_conflicts':E.route_reason.eq('conflict_to_stage2').to_numpy(),
      'support_seeker_unlabeled':E.role.eq('seeker').to_numpy(),
      'support_supporter_unlabeled':E.role.eq('supporter').to_numpy()}
    for name,mask in slices.items():
        if mask.any():SUBGROUPS.append(dict(**a,subgroup=name,**metric(y[mask],pr[mask])))

for item in ITEMS:
    if not item['selected']:continue
    arm,seed=item['arm'],item['seed'];v=item['selected']['variant'];tau=item['selected']['tau']
    p=PROBS[f'{arm}__{seed}__{v}'];fp=PROBS[f'{arm}__{seed}__fp32']
    for system in ['neural','hybrid_v2']:
        gate=get_gate(system);q=decide(p,tau,gate)
        for reference,rtau in [('same_threshold',tau),('original_default_threshold',PROTOCOL['default_tau'])]:
            r=decide(fp,rtau,gate)
            for name,idx in partition_indices.items():
                pos=idx[y[idx]==1];neg=idx[y[idx]==0]
                if not len(pos) and not len(neg):continue
                lost=((r[pos]==1)&(q[pos]==0)).astype(int)
                gained=((r[pos]==0)&(q[pos]==1)).astype(int)
                newfp=((r[neg]==0)&(q[neg]==1)).astype(int)
                k=int(lost.sum());n=len(pos);detected=int((r[pos]==1).sum())
                _,_,upper=exact_bounds(k,n)
                lo,hi=group_interval(lost,groups[pos],N_BOOT) if n else (None,None)
                FLIPS.append(dict(arm=arm,seed=seed,variant=v,system=system,partition=name,reference=reference,
                    quantized_tau=tau,fp32_tau=rtau,positive_lost=k,positive_gained=int(gained.sum()),
                    n_positive=n,fp32_detected_positive=detected,lost_rate=k/n if n else None,
                    conditional_loss_rate=k/detected if detected else None,lost_group_lo=lo,lost_group_hi=hi,
                    descriptive_binomial_upper95_per_seed=upper,new_negative_alerts=int(newfp.sum()),
                    absolute_quantized_recall=float(q[pos].mean()) if n else None))
    # Isolate routing with identical frozen quantized weights and threshold.
    neural=decide(p,tau);hybrid=decide(p,tau,E.gate_v2.to_numpy())
    for name,idx in partition_indices.items():
        for met,eligible in [('recall',y[idx]==1),('fpr',y[idx]==0),('alert_rate',np.ones(len(idx),dtype=bool))]:
            use=idx[eligible]
            if not len(use):continue
            delta=hybrid[use]-neural[use];lo,hi=group_interval(delta,groups[use],N_BOOT)
            ROUTING.append(dict(arm=arm,seed=seed,variant=v,partition=name,metric=met,n=len(use),
                hybrid_minus_neural=float(delta.mean()),group_lo=lo,group_hi=hi))
R=pd.DataFrame(RESULTS);F=pd.DataFrame(FLIPS)
R.to_csv(OUTPUT/'tables/external_results_per_seed.csv',index=False)
F.to_csv(OUTPUT/'tables/compression_flips_per_seed.csv',index=False)
pd.DataFrame(ROUTING).to_csv(OUTPUT/'tables/routing_same_weights_paired.csv',index=False)
pd.DataFrame(SUBGROUPS).to_csv(OUTPUT/'tables/predefined_public_subgroups.csv',index=False)
print('Primary predeclared model results:')
print(R[R.primary][['partition','recall','fpr','alert_rate','fn','fp','n_positive','n_negative','stage2_rate']].to_string(index=False))


## 7. Seed comparisons and threshold-free metrics
The selected-procedure summaries are conditional on development acceptance; the source acceptance table always retains rejected seeds. CE/SAQS comparisons of selected procedures use only their common accepted seeds and explicitly record that condition. The fixed R90 comparison uses all original seeds and saved thresholds, providing a less selection-conditioned comparison. R90 refers to an old calibration target, not a guaranteed recall on these new data.

Seed intervals quantify training-seed variability on fixed new texts. Example-group intervals quantify variation over observed text groups at fixed trained models. Neither establishes population clinical reliability, and multiple subgroup analyses are descriptive.


In [ ]:
summary=[]
for (arm,analysis,variant,system,part),g in R.groupby(['arm','analysis','variant','system','partition'],dropna=False):
    for met in ['recall','fpr','alert_rate','stage2_rate']:
        vals=pd.to_numeric(g[met],errors='coerce').dropna()
        if not len(vals):continue
        m,lo,hi=paired_seed_interval(vals)
        summary.append(dict(arm=arm,analysis=analysis,variant=variant,system=system,partition=part,metric=met,
            seeds=len(vals),mean=m,sd=float(vals.std(ddof=1)) if len(vals)>1 else None,
            seed_mean_lo=lo,seed_mean_hi=hi,
            conditioning='development-accepted seeds; do not compare unequal accepted groups as a paired experiment' if analysis=='source_selected' else 'all frozen source seeds'))
# Aggregate selected procedure across different development-selected variants per seed.
selected=R[R.analysis=='source_selected']
for (arm,system,part),g in selected.groupby(['arm','system','partition']):
    for met in ['recall','fpr','alert_rate','stage2_rate']:
        vals=pd.to_numeric(g[met],errors='coerce').dropna()
        if not len(vals):continue
        m,lo,hi=paired_seed_interval(vals)
        summary.append(dict(arm=arm,analysis='source_selected_procedure',variant='dev-selected-per-seed',system=system,
            partition=part,metric=met,seeds=len(vals),mean=m,sd=float(vals.std(ddof=1)) if len(vals)>1 else None,
            seed_mean_lo=lo,seed_mean_hi=hi,conditioning='development-accepted seeds only; rejected counts retained separately'))
pd.DataFrame(summary).to_csv(OUTPUT/'tables/external_seed_summary.csv',index=False)
paired=[]
if set(['ce','saqs'])<=set(ARMS):
    for analysis in ['source_selected','source_R90']:
        rr=R[R.analysis==analysis]
        for system in rr.system.unique():
            for part in rr.partition.unique():
                g=rr[(rr.system==system)&(rr.partition==part)]
                variants=['procedure'] if analysis=='source_selected' else sorted(g.variant.unique())
                for variant in variants:
                    gg=g if variant=='procedure' else g[g.variant==variant]
                    a=gg[gg.arm=='ce'];b=gg[gg.arm=='saqs']
                    for met in ['recall','fpr','alert_rate']:
                        pairs=a[['seed',met]].merge(b[['seed',met]],on='seed',suffixes=('_ce','_saqs')).dropna()
                        if not len(pairs):continue
                        m,lo,hi=paired_seed_interval(pairs[met+'_saqs']-pairs[met+'_ce'])
                        paired.append(dict(analysis=analysis,system=system,variant=variant,partition=part,metric=met,
                            paired_seeds=len(pairs),saqs_minus_ce=m,seed_delta_lo=lo,seed_delta_hi=hi,
                            conditioning='jointly development-accepted seeds' if analysis=='source_selected' else 'all available original paired seeds'))
pd.DataFrame(paired).to_csv(OUTPUT/'tables/ce_saqs_paired_seed_comparisons.csv',index=False)
thresholdfree=[]
for key,job in JOBS.items():
    for name,idx in partition_indices.items():
        valid=idx[np.isin(y[idx],[0,1])]
        if len(np.unique(y[valid]))<2:continue
        pp=PROBS[key][valid]
        thresholdfree.append(dict(job=key,arm=job['arm'],seed=job['seed'],variant=job['variant'],partition=name,
            auroc=float(roc_auc_score(y[valid],pp)),auprc=float(average_precision_score(y[valid],pp))))
pd.DataFrame(thresholdfree).to_csv(OUTPUT/'tables/threshold_free_discrimination.csv',index=False)
# Store per-row predictions without redistributing public source text.
rows=[]
for a in ANALYSES:
    ident=(a['arm'],a['seed'],a['variant'],a['analysis'],a['system'])
    pred=PREDICTIONS[ident]
    for i,row in E.iterrows():
        rows.append(dict(**a,source_id=row.source_id,text_hash=row.text_hash,group_id=row.group_id,partition=row.partition,
            label=int(row.label),probability=float(PROBS[a['job']][i]),prediction=int(pred[i]),route_reason=row.route_reason))
pd.DataFrame(rows).to_csv(OUTPUT/'tables/predictions_without_text.csv',index=False)
print('Tables saved. Source-development rejection rates must accompany selected-seed means.')


## 8. Export a compact report and integrity record
Models remain read-only in Kaggle Input. No archives contain weights or raw source texts. The report states historical-coverage limitations, target shortfalls, support-label limitations, and observed flips. It does not announce success or independence merely because the notebook completed.


In [ ]:
claims=[
 'FROZEN EXTERNAL EVALUATION: weights, rules, quantizers and thresholds were unchanged; no retraining or tuning used the public evaluation data.',
 'Historical overlap qualification: '+EVAL_PLAN['independence_status'],
 'Public Twitter labels are source text-classification labels, not clinician-rated individual risk.',
 'ESConv is unlabeled for this task: only alert rates are meaningful; never call them false-positive rates.',
 'The main test, probe and labeled hard negatives are disjoint sampled groups from one Twitter source, not three independently collected corpora.',
 'Public row labels and author history may be noisy/unknown. Duplicate groups do not establish independent authors.',
 'A zero empirical flip count is not a clinical-safety guarantee. Inspect absolute recall and fixed-threshold vs original-default comparisons separately.',
 'Seed means condition on source-development acceptance where specified; rejected seed counts are retained.',
 'R90 was selected on old calibration data and is not a target achieved by retuning on these new examples.',
 'No device latency claim follows from CPU inference in this notebook.'
]
for arm in ARMS:
    ss=[r for r in STATUS if r['arm']==arm];claims.append(f'{arm}: {sum(r["development_accepted"] for r in ss)}/{len(ss)} source seeds development-accepted; the remaining seeds were not replaced.')
if PRIMARY:
    for _,row in R[R.primary].iterrows():
        if row.n_labeled:
            claims.append(f'Primary {row.partition}: recall={row.recall if pd.notna(row.recall) else "n/a"}; dataset-label FPR={row.fpr if pd.notna(row.fpr) else "n/a"}; counts positive={row.n_positive}, negative={row.n_negative}.')
        else:claims.append(f'Primary {row.partition}: unlabeled alert rate={row.alert_rate}; n={row.n}.')
    pp=F[(F.arm==PRIMARY['arm'])&(F.seed==PRIMARY['seed'])&(F.variant==PRIMARY['variant'])&(F.system=='hybrid_v2')]
    for _,row in pp.iterrows():
        if row.n_positive:claims.append(f'Primary {row.partition}, reference={row.reference}: {row.positive_lost}/{row.n_positive} positive losses; do not pool seeds into independent trials.')
(OUTPUT/'tables/claim_check.txt').write_text('\n'.join(claims))
print('\n'.join(claims))
# Final content checks: input model/tokenizer bytes and frozen selection still match.
for key,job in JOBS.items():
    for relative,sha in job['files'].items():
        if file_sha(RUN/relative)!=sha:raise RuntimeError(f'Input artifact changed during evaluation: {relative}')
if digest(json.loads((RUN/'selection_frozen.json').read_text()))!=EXPECTED_SELECTION_HASH:
    raise RuntimeError('Frozen source selection changed')
atomic_json(OUTPUT/'manifests/evaluation_complete.json',dict(plan_hash=PLAN_HASH,
    evaluation_rows_hash=EVALUATION_HASH,model_bytes_reverified=True,new_models_trained=0,
    thresholds_reselected=0,result_rows=len(R),independence_status=EVAL_PLAN['independence_status']))
# Small allowlisted ZIP, written atomically and verified; no giant model duplication.
files=[]
for subdir in ['tables','manifests','predictions','figures']:
    files += [p for p in (OUTPUT/subdir).rglob('*') if p.is_file() and not p.is_symlink()]
needed=sum(p.stat().st_size for p in files)
import shutil
if shutil.disk_usage(OUTPUT).free<needed+128*1024**2:raise RuntimeError('Not enough space for the small report archive; raw result files remain saved.')
target=OUTPUT.parent/'crisisq_external_public_v1_reports.zip';tmp=target.with_suffix('.partial.zip')
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in files:z.write(p,p.relative_to(OUTPUT))
    z.writestr('READ_ME.txt','Inference-only external evaluation. See manifests/evaluation_frozen.json and historical_coverage.json. No raw source text or model weights included. No clinical validation claimed.\n')
with zipfile.ZipFile(tmp) as z:
    if z.testzip() is not None:raise RuntimeError('ZIP integrity check failed')
tmp.replace(target)
print('DOWNLOAD:',target,f'({target.stat().st_size/1e6:.1f} MB)')


### What this evaluation can and cannot settle

This supplies a reproducible, no-retuning check on new public sources without hiring annotators or clinicians. The existing source labels support task-level recall/FPR, and naturally occurring public texts can reveal routing failures. It does not manufacture expert-reviewed labels or independently authored compositional cases.

With latest-output-only input, full historical training/evaluation overlap may remain unverifiable. Report that explicitly to Reviewer 4; do not claim it fully satisfies their untouched-set request if the required historical reference archive is absent. Public source data may also have appeared in pretraining.

Do not use this run to find a favorable replacement seed, threshold, source, or keyword definition. If results disappoint, report them. Source restrictions and citation obligations remain those of the original repositories; this notebook does not redistribute their raw texts.

Technical documentation: [ONNX Runtime inference](https://onnxruntime.ai/docs/api/python/api_summary.html), [Transformers tokenizers](https://huggingface.co/docs/transformers/main_classes/tokenizer).
